In [6]:
# ============================================================
# ERIP - ENTERPRISE RISK INTELLIGENCE PLATFORM
# ============================================================
#
# Notebook
# --------
# nb_build_fact_stress_testing
#
# Layer
# -----
# Gold Layer - Fact Tables
#
# Purpose
# -------
# Build the loan-level stress testing fact table by applying
# macroeconomic scenario multipliers to Expected Credit Loss.
#
# Grain
# -----
# One row per loan per scenario.
#
# Output
# ------
# fact_stress_testing
#
# Enterprise Concepts
# -------------------
# ✓ Stress Testing
# ✓ Scenario Analysis
# ✓ IFRS 9
# ✓ Basel III Capital Planning
# ✓ Loan-level Risk Analytics
# ✓ Executive Risk Intelligence
# ============================================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime

ecl_fact_table = "fact_expected_credit_loss"
scenario_table = "dim_scenario"
date_table = "dim_date"

target_table = "fact_stress_testing"
pipeline_name = "nb_build_fact_stress_testing"

run_start_time = datetime.now()

print("ERIP Fact Stress Testing Build Started")

StatementMeta(, be71a023-7e81-4541-98a5-585352df7007, 8, Finished, Available, Finished, False)

ERIP Fact Stress Testing Build Started


In [7]:
# ============================================================
# SECTION 2 - READ SOURCE TABLES
# ============================================================

fact_expected_credit_loss = spark.table("fact_expected_credit_loss")
scenario_set = spark.table("dim_scenario")
dim_date = spark.table("dim_date")

print(f"ECL Fact Rows : {fact_expected_credit_loss.count()}")
print(f"Scenario Rows : {scenario_set.count()}")
print(f"Date Rows     : {dim_date.count()}")

StatementMeta(, be71a023-7e81-4541-98a5-585352df7007, 9, Finished, Available, Finished, False)

ECL Fact Rows : 5000
Scenario Rows : 108
Date Rows     : 36


In [8]:
# ============================================================
# SECTION 3 - LOAD MONTHLY SCENARIO DIMENSION
# ============================================================
#
# Purpose
# -------
# Use the complete monthly scenario dimension.
#
# Enterprise Design
# -----------------
# 36 Months
# ×
# Baseline
# Adverse
# Severe
#
# = 108 scenario records
#
# Each loan will be stress tested across every month
# and every macroeconomic scenario.
# ============================================================

scenario_set = spark.table("dim_scenario")

print(f"Scenario Records : {scenario_set.count()}")

display(scenario_set.limit(10))

StatementMeta(, be71a023-7e81-4541-98a5-585352df7007, 10, Finished, Available, Finished, False)

Scenario Records : 108


SynapseWidget(Synapse.DataFrame, c86a2927-337e-4b3e-a165-ada3b6314bc8)

In [9]:
# ============================================================
# SECTION 4 - BUILD FACT STRESS TESTING
# ============================================================
#
# Purpose
# -------
# Create loan-level monthly scenario results by applying
# macroeconomic PD and LGD stress multipliers to the
# Expected Credit Loss fact table.
#
# Grain
# -----
# One row per loan per scenario per month.
#
# Expected Row Count
# ------------------
# 5,000 loans × 108 scenario-month records = 540,000 rows
# ============================================================

fact_stress_testing = (
    fact_expected_credit_loss.alias("e")
    .crossJoin(scenario_set.alias("s"))
    .withColumn(
        "stressed_pd",
        least(
            col("e.pd") * col("s.pd_stress_multiplier"),
            lit(1.0)
        )
    )
    .withColumn(
        "stressed_lgd",
        least(
            col("e.lgd") * col("s.lgd_stress_multiplier"),
            lit(1.0)
        )
    )
    .withColumn(
        "stressed_ecl",
        col("e.exposure_at_default") * col("stressed_pd") * col("stressed_lgd")
    )
    .withColumn(
        "ecl_increase_amount",
        col("stressed_ecl") - col("e.calculated_ecl")
    )
    .withColumn(
        "ecl_increase_pct",
        when(
            col("e.calculated_ecl") > 0,
            (col("ecl_increase_amount") / col("e.calculated_ecl")) * 100
        ).otherwise(0)
    )
    .withColumn(
        "capital_impact_estimate",
        col("ecl_increase_amount") * lit(1.25)
    )
    .select(
        col("e.loan_sk"),
        col("e.loan_id"),
        col("e.facility_id"),
        col("e.customer_sk"),
        col("e.customer_id"),
        col("e.country_sk"),
        col("e.industry_sk"),
        col("e.rating_sk"),

        # Scenario dimension keys and attributes
        col("s.macro_sk").alias("scenario_sk"),
        col("s.scenario_id"),
        col("s.scenario_name"),
        col("s.scenario_rank"),
        col("s.scenario_severity"),
        col("s.scenario_month"),
        col("s.stress_intensity"),
        col("s.pd_stress_multiplier"),
        col("s.lgd_stress_multiplier"),

        # Base exposure and risk measures
        col("e.exposure_at_default"),
        col("e.pd").alias("base_pd"),
        col("e.lgd").alias("base_lgd"),
        col("e.calculated_ecl").alias("base_ecl"),

        # Stressed measures
        col("stressed_pd"),
        col("stressed_lgd"),
        col("stressed_ecl"),
        col("ecl_increase_amount"),
        col("ecl_increase_pct"),
        col("capital_impact_estimate"),

        current_timestamp().alias("gold_updated_timestamp")
    )
)

print(f"Stress testing fact rows created: {fact_stress_testing.count()}")
display(fact_stress_testing.limit(10))

StatementMeta(, be71a023-7e81-4541-98a5-585352df7007, 11, Finished, Available, Finished, False)

Stress testing fact rows created: 540000


SynapseWidget(Synapse.DataFrame, 655abc80-a6a1-4d94-8cbe-536bbcf6cf69)

In [10]:
# ============================================================
# SECTION 5 - FACT STRESS TESTING QUALITY VALIDATION
# ============================================================

total_rows = fact_stress_testing.count()

expected_rows = fact_ecl.count() * scenario_set.count()

null_loan_sk = fact_stress_testing.filter(col("loan_sk").isNull()).count()

null_customer_sk = fact_stress_testing.filter(col("customer_sk").isNull()).count()

null_scenario_sk = fact_stress_testing.filter(col("scenario_sk").isNull()).count()

invalid_stressed_pd = fact_stress_testing.filter(
    (col("stressed_pd") < 0) | (col("stressed_pd") > 1)
).count()

invalid_stressed_lgd = fact_stress_testing.filter(
    (col("stressed_lgd") < 0) | (col("stressed_lgd") > 1)
).count()

negative_stressed_ecl = fact_stress_testing.filter(
    col("stressed_ecl") < 0
).count()

print("Fact Stress Testing Quality Checks")
print("----------------------------------")
print(f"Rows                     : {total_rows}")
print(f"Expected Rows            : {expected_rows}")
print(f"Null Loan SK             : {null_loan_sk}")
print(f"Null Customer SK         : {null_customer_sk}")
print(f"Null Scenario SK         : {null_scenario_sk}")
print(f"Invalid Stressed PD      : {invalid_stressed_pd}")
print(f"Invalid Stressed LGD     : {invalid_stressed_lgd}")
print(f"Negative Stressed ECL    : {negative_stressed_ecl}")

if (
    total_rows != expected_rows or
    null_loan_sk > 0 or
    null_customer_sk > 0 or
    null_scenario_sk > 0 or
    invalid_stressed_pd > 0 or
    invalid_stressed_lgd > 0 or
    negative_stressed_ecl > 0
):
    raise Exception("Fact Stress Testing Validation Failed")
else:
    print("✓ Fact Stress Testing Validation Passed")

StatementMeta(, be71a023-7e81-4541-98a5-585352df7007, 12, Finished, Available, Finished, False)

Fact Stress Testing Quality Checks
----------------------------------
Rows                     : 540000
Expected Rows            : 540000
Null Loan SK             : 0
Null Customer SK         : 0
Null Scenario SK         : 0
Invalid Stressed PD      : 0
Invalid Stressed LGD     : 0
Negative Stressed ECL    : 0
✓ Fact Stress Testing Validation Passed


In [12]:
# ============================================================
# SECTION 6 - WRITE GOLD FACT TABLE
# ============================================================

(
    fact_stress_testing.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(target_table)
)

print(f"✓ Gold fact table created: {target_table}")
print(f"Rows written: {fact_stress_testing.count()}")

StatementMeta(, be71a023-7e81-4541-98a5-585352df7007, 14, Finished, Available, Finished, False)

✓ Gold fact table created: fact_stress_testing
Rows written: 540000
